In [7]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
%matplotlib inline
import matplotlib
matplotlib.rcParams["figure.figsize"]=(20,10)
import seaborn as sns;sns.set()
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px
from datetime import datetime,time
from plotly.offline import iplot,plot,init_notebook_mode,download_plotlyjs
%matplotlib inline 
init_notebook_mode(connected=True)

---

In [8]:
from scipy import stats
def more_stats(df,numerical_cols):
    from scipy import stats
    confidence_level = 0.95
    confidence_interval_list=[]

    for i in numerical_cols:
        sample_mean = np.mean(df[i])
        standard_error_of_mean = stats.sem(df[i])
        degrees_of_freedom = len(df[i]) - 1
        confidence_interval = stats.t.interval(
            df=degrees_of_freedom,
            loc=sample_mean,
            scale=standard_error_of_mean,
            confidence=confidence_level
        )
        
        confidence_interval_list.append(confidence_interval)
    ci_df = pd.DataFrame(confidence_interval_list, columns=['CI_lower', 'CI_upper'])
    ci_df['column'] = numerical_cols
    ci_df = ci_df[['column', 'CI_lower', 'CI_upper']]

    return  pd.concat([df[numerical_cols].var().to_frame().rename(columns={0:"variance"}),
            df[numerical_cols].std().to_frame().rename(columns={0:"std"}),
            df[numerical_cols].skew().to_frame().rename(columns={0:"skewness"}),
            df[numerical_cols].kurtosis().to_frame().rename(columns={0:"kurtosis"}),
            df[numerical_cols].sem().to_frame().rename(columns={0:"st error of the mean"}),
            ci_df.set_index("column")
            ],axis=1)

---

In [9]:
import numpy as np
import pandas as pd
from scipy.stats import lognorm, norm, gamma

def price_conditional_distribution_prob_all(
    df, price_col, threshold, category_col=None, dist="lognormal"
):
    """
    Compute all conditional probabilities using a fitted distribution.
    Calculates: P(>), P(<), P(=), P(>=), P(<=)
    Category column (if provided) is shown first in the output.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset containing price column and optional categorical column.
    price_col : str
        Numeric column to fit distribution on.
    threshold : float
        Threshold value for probability calculation.
    category_col : str, optional
        Categorical column name to condition on.
    dist : str
        Distribution type: 'lognormal', 'normal', or 'gamma'.

    Returns
    -------
    pd.DataFrame
        Probabilities per category (or overall if no category).
    """

    valid_dists = ["lognormal", "normal", "gamma"]
    if dist not in valid_dists:
        raise ValueError(f"Invalid distribution '{dist}'. Must be one of {valid_dists}")

    results = []

    # --- helper function ---
    def compute_probs(prices):
        # Fit the chosen distribution
        if dist == "lognormal":
            shape, loc, scale = lognorm.fit(prices, floc=0)
            cdf_val = lognorm.cdf(threshold, shape, loc=loc, scale=scale)
        elif dist == "normal":
            mu, sigma = norm.fit(prices)
            cdf_val = norm.cdf(threshold, mu, sigma)
        elif dist == "gamma":
            shape, loc, scale = gamma.fit(prices, floc=0)
            cdf_val = gamma.cdf(threshold, shape, loc=loc, scale=scale)

        # Convert to probabilities for all comparison operators
        probs = {
            f"P({price_col} > {threshold})": 1 - cdf_val,
            f"P({price_col} < {threshold})": cdf_val,
            f"P({price_col} >= {threshold})": 1 - cdf_val,
            f"P({price_col} <= {threshold})": cdf_val,
            f"P({price_col} = {threshold})": 0.0,  # continuous dist → 0
        }
        return probs

    # --- no categorical condition ---
    if category_col is None:
        prices = df[price_col]
        probs = compute_probs(prices)
        # Make category column for consistency
        probs = {"Condition": "All", **probs}
        results.append(probs)
        df_result = pd.DataFrame(results)
        df_result = df_result[
            ["Condition"]
            + [col for col in df_result.columns if col != "Condition"]
        ]
        return df_result

    # --- with categorical condition ---
    for cat, group in df.groupby(category_col):
        prices = group[price_col]
        probs = compute_probs(prices)
        # Make sure category_col is first
        probs = {category_col: cat, **probs}
        results.append(probs)

    df_result = pd.DataFrame(results)
    df_result = df_result[
        [category_col]
        + [col for col in df_result.columns if col != category_col]
    ]
    return df_result


---

In [10]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import lognorm, norm, gamma

def plot_price_distribution_fit(df, price_col, threshold, category_col=None, dist="lognormal"):
    """
    Plot histogram, fitted distribution density (KDE), and ECDF.
    """
    plt.figure(figsize=(20, 5))

    # 1️⃣ Histogram
    plt.subplot(2, 3, 4)
    sns.histplot(
        data=df,
        x=price_col,
        hue=category_col,
        bins=50,
        kde=False,
        multiple='stack',
        alpha=0.7
    )
    plt.axvline(threshold, color='red', linestyle='--', label=f"Threshold = {threshold}")
    plt.legend()
    plt.title(f"{price_col} histogram by {category_col if category_col else 'All'}")

    # 2️⃣ Fitted density (KDE approximation)
    plt.subplot(2, 3, 5)
    if dist == "lognormal":
        shape, loc, scale = lognorm.fit(df[price_col], floc=0)
        x_vals = np.linspace(df[price_col].min(), df[price_col].max(), 1000)
        plt.plot(x_vals, lognorm.pdf(x_vals, shape, loc=loc, scale=scale), color='blue')
    elif dist == "normal":
        mu, sigma = norm.fit(df[price_col])
        x_vals = np.linspace(df[price_col].min(), df[price_col].max(), 1000)
        plt.plot(x_vals, norm.pdf(x_vals, mu, sigma), color='blue')
    elif dist == "gamma":
        shape, loc, scale = gamma.fit(df[price_col], floc=0)
        x_vals = np.linspace(df[price_col].min(), df[price_col].max(), 1000)
        plt.plot(x_vals, gamma.pdf(x_vals, shape, loc=loc, scale=scale), color='blue')
    plt.axvline(threshold, color='red', linestyle='--')
    plt.title(f"{price_col} fitted {dist} density by {category_col if category_col else 'All'}")

    # 3️⃣ ECDF
    plt.subplot(2, 3, 6)
    sns.ecdfplot(
        data=df,
        x=price_col,
        hue=category_col
    )
    plt.axvline(threshold, color='red', linestyle='--')
    plt.title(f"{price_col} cumulative density by {category_col if category_col else 'All'}")

    plt.tight_layout()
    plt.show()


---

In [11]:
import numpy as np
import pandas as pd

def price_conditional_empirical_prob_all(df, price_col, threshold, category_col=None):
    """
    Compute all empirical conditional probabilities for a numeric variable:
    P(>), P(<), P(=), P(>=), P(<=)
    
    Optionally conditioned on a categorical column.
    The category column (if provided) appears first in the output.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset containing the data.
    price_col : str
        Column name of the numeric variable (e.g. 'price').
    threshold : float or int
        Threshold value for the probability calculation.
    category_col : str, optional
        Categorical column name for grouping (e.g. 'segment').

    Returns
    -------
    pd.DataFrame
        A DataFrame with empirical probabilities (by group if provided).
    """
    results = []

    # Helper function to compute all probabilities for a numeric Series
    def compute_probs(values):
        probs = {
            f"P({price_col} > {threshold})": np.mean(values > threshold),
            f"P({price_col} < {threshold})": np.mean(values < threshold),
            f"P({price_col} = {threshold})": np.mean(values == threshold),
            f"P({price_col} >= {threshold})": np.mean(values >= threshold),
            f"P({price_col} <= {threshold})": np.mean(values <= threshold),
        }
        return probs

    # No category condition → overall probabilities
    if category_col is None:
        prices = df[price_col]
        probs = compute_probs(prices)
        probs = {"Condition": "All", **probs}
        results.append(probs)
        df_result = pd.DataFrame(results)
        df_result = df_result[
            ["Condition"]
            + [col for col in df_result.columns if col != "Condition"]
        ]
        return df_result

    # With category condition → probabilities per group
    for cat, group in df.groupby(category_col):
        prices = group[price_col]
        probs = compute_probs(prices)
        probs = {category_col: cat, **probs}
        results.append(probs)

    df_result = pd.DataFrame(results)
    df_result = df_result[
        [category_col]
        + [col for col in df_result.columns if col != category_col]
    ]
    return df_result


---

In [12]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_price_distributions(df, price_col, threshold, category_col=None):
    """
    Visualize histogram, density, and ECDF of a numeric column.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Original dataset.
    price_col : str
        Numeric column to visualize.
    threshold : float or int
        Threshold line to add in the plots.
    category_col : str, optional
        Categorical column for grouping (hue).
    """
    plt.figure(figsize=(20, 5))

    # 1️⃣ Histogram
    plt.subplot(2, 3, 4)
    sns.histplot(
        data=df,
        x=price_col,
        hue=category_col,
        bins=50,
        kde=False,
        multiple='stack',
        alpha=0.7
    )
    plt.axvline(threshold, color='red', linestyle='--', label=f'Threshold = {threshold}')
    plt.legend()
    plt.title(f"{price_col} histogram by {category_col if category_col else 'All'}")

    # 2️⃣ Density (KDE)
    plt.subplot(2, 3, 5)
    sns.kdeplot(
        data=df,
        x=price_col,
        hue=category_col,
        fill=True
    )
    plt.axvline(threshold, color='red', linestyle='--')
    plt.title(f"{price_col} density function by {category_col if category_col else 'All'}")

    # 3️⃣ ECDF
    plt.subplot(2, 3, 6)
    sns.ecdfplot(
        data=df,
        x=price_col,
        hue=category_col
    )
    plt.axvline(threshold, color='red', linestyle='--')
    plt.title(f"{price_col} cumulative density by {category_col if category_col else 'All'}")

    plt.tight_layout()
    plt.show()


In [13]:
from scipy.stats import spearmanr,mannwhitneyu
def mannwhitneyu_func(Dataset:pd.DataFrame,Numericaltarget:str,BinaryFeature:str):
    group1 = Dataset[Numericaltarget][Dataset[BinaryFeature] == 0]
    group2 = Dataset[Numericaltarget][Dataset[BinaryFeature]  == 1]
    u_stat, p_mw = mannwhitneyu(group1, group2, alternative='two-sided')
    print(f"W = {u_stat:.3f}, p = {p_mw:.3f}")
    if p_mw < 0.05:
        print(f"As The P-value is less than .05 : Reject H₀ → There is significant distribution difference between {Numericaltarget} & {BinaryFeature} (typically medians)")
    else:
        print(f"As The P-value is more than .05 : Fail to reject H₀ → No significant distribution difference between {Numericaltarget} & {BinaryFeature} (typically medians)")

    stat, p = mannwhitneyu(group1, group2, alternative='greater')    
    if p < 0.05 :
        print(f"more over this suggest that the group 1 {BinaryFeature} ==0 () tend to have higher price values than group 2 {BinaryFeature} ==1")   
    else:
        print(f"more over this suggest that the group 2 {BinaryFeature} ==1 () tend to have higher price values than group 1 {BinaryFeature} ==0")     

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    sns.boxplot(x=BinaryFeature, y=Numericaltarget, data=Dataset, palette="Set2",ax=axes[0])
    sns.stripplot(x=BinaryFeature, y=Numericaltarget, data=Dataset, color="black", alpha=0.6,ax=axes[1]);
    sns.histplot(x=Numericaltarget,hue=BinaryFeature,ax=axes[2],data=Dataset,alpha=0.6)

In [14]:
def spearmanr_func(Dataset:pd.DataFrame,Numericaltarget:str,OrdinalFeature:str):
    spearman_corr, p_spear = spearmanr(Dataset[Numericaltarget],Dataset[OrdinalFeature])
    prob_df = Dataset.groupby(OrdinalFeature)[Numericaltarget].mean().reset_index()
    prob_df.rename(columns={Numericaltarget: 'prob_target_1'}, inplace=True)  

    print(f"spearman_corr = {spearman_corr:.3f}, p = {p_spear:.3f}")
    if p_spear < 0.05:
        print(f"As The P-value is less than .05 : Reject H₀ → There is significant monotonic relationship between {Numericaltarget} & {OrdinalFeature}")
    else:
        print(f"As The P-value is more than .05 : Fail to reject H₀ → No significant monotonic relationship between {Numericaltarget} & {OrdinalFeature}")

    if spearman_corr > 0:
        print(f"more over this suggest that the As the {OrdinalFeature} variable increases, the {Numericaltarget} variable tends to increase (positive association).")
    else:
        print(f"more over this suggest that the As the {OrdinalFeature} variable increases, the {Numericaltarget} variable tends to decrease (negative association).")    
            

    plt.figure(figsize=(20,5))
    ax=sns.heatmap(prob_df.set_index(OrdinalFeature).T, annot=True, cmap="YlGnBu", cbar=True)
    ax.set_title("Probability of Binary Target=1 by Ordinal Feature")
    ax.set_xlabel("Ordinal Feature")
    ax.set_ylabel("Probability")
    plt.tight_layout()
    plt.show()  

In [17]:
import scikit_posthocs as sp
def posthoc_dunn_test(Dataset,Numericaltarget,CategoricalFeature):
    import scikit_posthocs as sp
    posthoc = sp.posthoc_dunn(Dataset, val_col=Numericaltarget, group_col=CategoricalFeature, p_adjust='bonferroni')
    plt.figure(figsize=(20,7))
    ax=sns.heatmap(posthoc.round(2),annot=True,cmap="Reds")
    ax.set_title(f"posthoc_dunn test -p_adjust- between {Numericaltarget} and {CategoricalFeature} groups")
    print("if p_adjust < .05 ==> significant difference between those two groups.")

In [16]:
def kruskal_func(Dataset:pd.DataFrame,Numericaltarget:str,CategoricalFeature:str):
    from scipy.stats import kruskal
    groups = [Dataset[Dataset[CategoricalFeature] == level][Numericaltarget] for level in Dataset[CategoricalFeature].unique()]

    stat, p = kruskal(*groups)
    if p < 0.05: 
        print(f"Kruskal-Wallis Test: Statistic={stat:.3f}, P-value={p:.3f} ==>\n The p value less than 0.05 ==> At least one group has Significant difference")
    else:
        print(f"Kruskal-Wallis Test: Statistic={stat:.3f}, P-value={p:.3f} ==>\n the p value more than 0.05 ==> No Significant difference between groups")

    fig, axes = plt.subplots(1, 2, figsize=(20, 5))
    sns.boxplot(x=CategoricalFeature, y=Numericaltarget, data=Dataset, palette="Set2",ax=axes[0])
    sns.stripplot(x=CategoricalFeature, y=Numericaltarget, data=Dataset, color="black", alpha=0.6, jitter=True,ax=axes[1])
    plt.title(f"Kruskal–Wallis Test\nH={stat:.2f}, p={p:.4f}")
    plt.show()

    if p < 0.05:
        print("As At least one group has Significant difference --> posthoc_dunn test")
        posthoc_dunn_test(Dataset,Numericaltarget,CategoricalFeature)

In [ ]:
%matplotlib inline 
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def feature_development(Dataset,Numericaltarget,rolling_window,span_n):
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    fig = plt.figure(figsize=(20, 8))
    gs = gridspec.GridSpec(2, 2, height_ratios=[1, 1])

    ax1 = fig.add_subplot(gs[0, 0])
    Dataset[Numericaltarget].rolling(window=rolling_window).mean().plot(color="red")
    ax1.set_title(f"Price SMA - Simple Moving Average (yearly) window= {rolling_window}")

    ax2 = fig.add_subplot(gs[0, 1])
    Dataset[Numericaltarget].expanding().mean().plot(color="red")
    ax2.set_title("Price CMA - Cumulative Moving Avearge (yearly)")

    ax3 = fig.add_subplot(gs[1, 0])
    Dataset[Numericaltarget].ewm(alpha=.01,adjust=False).mean().plot(color="red")
    ax3.set_title("Price EMA - Exponential Moving Average (yearly)")


    ax4 = fig.add_subplot(gs[1, 1])
    Dataset[Numericaltarget].ewm(span=span_n).mean().plot(color="red")
    ax4.set_title(f"Price EMWA - Exponential Moving Weighted Average span= {span_n} -- this the is best method to use when it comes to\n times series analysus as this one gives weight for the most recent data values (yearly)");

    plt.tight_layout()
    plt.show()
